# RAG Codegen AOSP — LLM-based (Full Run v7)
**Purpose:** Run the complete 4-condition experiment: C1 → C2 → ChromaDB → DSPy → C3 (RAG+DSPy) → C4 (Feedback).

**Runtime:** Colab A100 High-RAM  
**Model:** Qwen 2.5-coder:32b via Ollama

---
### Pipeline overview
| # | Phase | Script | Est. time |
|---|-------|--------|-----------|
| 1–5 | Setup | Drive, deps, Ollama, repo | 10 min |
| 6 | C1 Baseline | `multi_main.py` | 30 min |
| 7 | C2 Adaptive | `multi_main_adaptive.py` | 30 min |
| 8 | ChromaDB | `rag.aosp_indexer` | 5 min (or 30s from cache) |
| 9 | DSPy Optimiser | `dspy_opt/optimizer.py` | 2–3 hrs (train-size 8) |
| 10 | Restore + Preflight | — | 1 min |
| 11 | Bug fixes | — | instant |
| 12 | C3 RAG+DSPy | `multi_main_rag_dspy.py` | 20 min |
| 13 | C4 Feedback | `multi_main_c4_feedback.py` | 20 min |
| 14 | Reporting | diagnose → rescore → compare | 2 min |
| 15 | Export | — | 1 min |

### Execution order rationale
> ChromaDB **must** be built before DSPy so the optimizer bootstraps traces
> with RAG context. DSPy prompts optimised without RAG underperform in C3/C4.

### Changes in v7 (vs v6)
- `optuna` added to pip install (required by MIPROv2 Bayesian search)
- `OLLAMA_NUM_PARALLEL=4` for faster DSPy optimization and generation
- DSPy default train-size bumped to 8 for thesis-quality results
- Time estimates added to pipeline overview

## 1. Configuration

In [ ]:
# ── Configuration — edit these paths once ─────────────────────────
import os, time, shutil, glob, subprocess, requests
from pathlib import Path

DRIVE_SRC    = "/content/drive/MyDrive/mse25_thesis"
REPO_URL     = "https://github.com/appdev1307/code-codegen-aosp-llm-based.git"
REPO_DIR     = "/content/code-codegen-aosp-llm-based"
MODEL_NAME   = "qwen2.5-coder:32b"
OLLAMA_LOG   = "/content/ollama.log"

# AOSP source & ChromaDB
AOSP_SRC_DIR = f"{REPO_DIR}/aosp_source"
CHROMA_DB    = f"{REPO_DIR}/rag/chroma_db"
CHROMA_ZIP   = f"{DRIVE_SRC}/chroma_db.zip"

# Restore mappings
RESTORE_MAP = {
    "output_c1.zip":          f"{REPO_DIR}/output",
    "output_c2.zip":          f"{REPO_DIR}/output_adaptive",
    "dspy_saved_backup.zip":  f"{REPO_DIR}/dspy_opt/saved",
}

# Ollama parallelism — uses more VRAM but speeds up DSPy 2-3x
os.environ["OLLAMA_NUM_PARALLEL"] = "4"

print("✓ Config loaded (OLLAMA_NUM_PARALLEL=4)")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
Path(DRIVE_SRC).mkdir(parents=True, exist_ok=True)
print(f"✓ Drive mounted — {DRIVE_SRC}")

## 3. System dependencies

In [ ]:
%%capture
!apt-get update -qq
!apt-get install -y -qq clang checkpolicy zstd

for tool in ["clang", "checkpolicy", "zstd"]:
    path = subprocess.run(["which", tool], capture_output=True, text=True).stdout.strip()
    print(f"  {tool}: {'✓ ' + path if path else '❌ NOT FOUND'}")

## 4. Ollama setup

In [ ]:
def start_ollama():
    """Install (if needed), start Ollama with parallel support, wait until healthy."""
    if not Path("/usr/local/bin/ollama").exists():
        print("Installing Ollama...")
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, capture_output=True)

    # Kill any existing Ollama process to pick up new env vars
    subprocess.run(["pkill", "-f", "ollama serve"], capture_output=True)
    time.sleep(1)

    subprocess.Popen(["ollama", "serve"],
                     stdout=open(OLLAMA_LOG, "w"), stderr=subprocess.STDOUT,
                     env={**os.environ})
    for i in range(30):
        try:
            r = requests.get("http://localhost:11434/api/tags", timeout=2)
            if r.status_code == 200:
                print(f"✓ Ollama server ready (took {i+1}s, parallel={os.environ.get('OLLAMA_NUM_PARALLEL', '1')})")
                return True
        except Exception:
            pass
        time.sleep(1)
    raise RuntimeError("❌ Ollama failed to start — check ollama.log")

start_ollama()

In [ ]:
# Pull model (skip if already cached)
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
if MODEL_NAME.split(":")[0] in result.stdout:
    print(f"✓ Model {MODEL_NAME} already available")
else:
    print(f"Pulling {MODEL_NAME}...")
    !ollama pull {MODEL_NAME}
!ollama list

## 5. Clone & setup repository

In [ ]:
if Path(REPO_DIR).exists():
    print("Repo exists — pulling latest...")
    subprocess.run(["git", "stash"], cwd=REPO_DIR, capture_output=True)
    subprocess.run(["git", "pull", "origin", "main"], cwd=REPO_DIR)
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print(f"✓ Working directory: {os.getcwd()}")

In [ ]:
%%capture
!pip install -q -r requirements.txt
!pip install -q requests pyyaml chromadb sentence-transformers dspy-ai
!pip install -q jinja2 fastapi uvicorn pydantic
!pip install -q optuna
print("✓ Dependencies installed (including optuna for MIPROv2)")

## 6. Run C1 — Baseline
> Vanilla LLM code generation (~30 min).

In [ ]:
start_ollama()
!python multi_main.py

In [ ]:
shutil.make_archive(f"{DRIVE_SRC}/output_c1", "zip", f"{REPO_DIR}/output")
print(f"✓ C1 saved to Drive ({len(list(Path(f'{REPO_DIR}/output').rglob('*')))} files)")

## 7. Run C2 — Adaptive
> Adaptive generation with RL-based prompt selection (~30 min).
> Produces `VSS_LABELLED_50.json` needed by DSPy.

In [ ]:
start_ollama()
!python multi_main_adaptive.py

In [ ]:
shutil.make_archive(f"{DRIVE_SRC}/output_c2", "zip", f"{REPO_DIR}/output_adaptive")
print(f"✓ C2 saved to Drive ({len(list(Path(f'{REPO_DIR}/output_adaptive').rglob('*')))} files)")

## 8. Build ChromaDB from AOSP source (or restore from cache)
> **Must run before DSPy.** ~5 min fresh, ~30s from Drive cache.

In [ ]:
# ── 8a. Try restoring from Drive ──────────────────────────────────
chroma_restored = False
if Path(CHROMA_ZIP).exists():
    print(f"Found cached ChromaDB: {CHROMA_ZIP}")
    target = Path(CHROMA_DB)
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(CHROMA_ZIP, CHROMA_DB)
    chroma_restored = True
    print("✓ ChromaDB restored from Drive cache")
else:
    print("⚠ No cached ChromaDB — will build from AOSP source")

In [ ]:
# ── 8b. Clone AOSP (only if not restored) ────────────────────────
if not chroma_restored:
    print("Cloning AOSP source (shallow, ~300 MB)...\n")
    aosp_repos = [
        ("https://android.googlesource.com/platform/hardware/interfaces", "hardware"),
        ("https://android.googlesource.com/platform/system/sepolicy",     "sepolicy"),
        ("https://android.googlesource.com/platform/packages/services/Car", "car"),
    ]
    Path(AOSP_SRC_DIR).mkdir(parents=True, exist_ok=True)
    for url, name in aosp_repos:
        dest = f"{AOSP_SRC_DIR}/{name}"
        if Path(dest).exists():
            print(f"  ✓ {name} already cloned")
            continue
        print(f"  Cloning {name}...")
        r = subprocess.run(["git", "clone", "--depth=1", url, dest], capture_output=True, text=True)
        print(f"  {'✓' if r.returncode == 0 else '❌'} {name}")
    print("\n✓ AOSP source ready")
else:
    print("Skipping — ChromaDB restored from cache")

In [ ]:
# ── 8c. Index AOSP → ChromaDB ────────────────────────────────────
if not chroma_restored:
    print("Running AOSP indexer...\n")
    !python -m rag.aosp_indexer --source {AOSP_SRC_DIR} --db {CHROMA_DB}
    print("\n✓ Indexing complete")
else:
    print("Skipping — ChromaDB already populated")

In [ ]:
# ── 8d. Verify ChromaDB ──────────────────────────────────────────
import chromadb
client = chromadb.PersistentClient(path=CHROMA_DB)
collections = client.list_collections()
total_chunks = sum(col.count() for col in collections)
print(f"  Collections: {len(collections)}, Total chunks: {total_chunks}")
for col in collections:
    print(f"    → {col.name}: {col.count()}")
assert total_chunks > 0, "❌ ChromaDB is EMPTY!"
print(f"\n✓ ChromaDB verified")
del client

In [ ]:
# ── 8e. Save to Drive ────────────────────────────────────────────
if not chroma_restored and not Path(CHROMA_ZIP).exists():
    shutil.make_archive(CHROMA_ZIP.replace(".zip", ""), "zip", CHROMA_DB)
    print(f"✓ Saved to {CHROMA_ZIP}")
else:
    print("ChromaDB cache exists — skipping save")

## 9. Run DSPy optimiser (MIPROv2)
> Bayesian search (TPE via Optuna) over prompt instructions + few-shot demos.
> Runs **after** ChromaDB so traces include RAG context.
> `train-size 8` for thesis-quality results (~2-3 hrs with parallel=4).

In [ ]:
start_ollama()

# Thesis run: train-size 8 (best quality, ~2-3 hrs)
!python dspy_opt/optimizer.py --mipro-auto light --train-size 8 --force

# Faster alternatives (uncomment one):
#!python dspy_opt/optimizer.py --mipro-auto light --train-size 4 --force   # ~1 hr
#!python dspy_opt/optimizer.py --mipro-auto light --train-size 2 --force   # ~20 min

dspy_count = len(glob.glob("dspy_opt/saved/*/program.json"))
print(f"\n✓ DSPy programs: {dspy_count}/12")

In [ ]:
shutil.make_archive(f"{DRIVE_SRC}/dspy_saved_backup", "zip", f"{REPO_DIR}/dspy_opt/saved")
print("✓ DSPy programs saved to Drive")

## 10. Restore cached outputs & preflight check
> Restores C1/C2/DSPy from Drive if not on disk (e.g. after Colab restart).

In [ ]:
# ── 10a. Restore if missing ───────────────────────────────────────
for zip_name, target_dir in RESTORE_MAP.items():
    src_path = f"{DRIVE_SRC}/{zip_name}"
    target = Path(target_dir)
    if target.exists() and len(list(target.rglob("*"))) > 0:
        print(f"  ✓ {zip_name}: on disk")
        continue
    if not Path(src_path).exists():
        print(f"  ⚠ {zip_name}: not on Drive")
        continue
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(src_path, target_dir)
    print(f"  ✓ {zip_name}: restored")

print(f"\n  DSPy programs: {len(glob.glob(f'{REPO_DIR}/dspy_opt/saved/*/program.json'))}/12")

In [ ]:
# ── 10b. Preflight ───────────────────────────────────────────────
errors = []

try:
    r = requests.get("http://localhost:11434/api/tags", timeout=2)
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"  ✓ Ollama: {MODEL_NAME}" if any(MODEL_NAME.split(":")[0] in m for m in models) else "")
    if not any(MODEL_NAME.split(":")[0] in m for m in models):
        errors.append(f"Model {MODEL_NAME} not found")
except Exception:
    errors.append("Ollama not responding")

try:
    client = chromadb.PersistentClient(path=CHROMA_DB)
    chunks = sum(c.count() for c in client.list_collections())
    del client
    print(f"  ✓ ChromaDB: {chunks} chunks") if chunks > 0 else errors.append("ChromaDB empty")
except Exception as e:
    errors.append(f"ChromaDB: {e}")

dspy_n = len(glob.glob(f"{REPO_DIR}/dspy_opt/saved/*/program.json"))
print(f"  ✓ DSPy: {dspy_n} programs")

for label, path in [("C1", f"{REPO_DIR}/output"), ("C2", f"{REPO_DIR}/output_adaptive")]:
    if Path(path).exists() and len(list(Path(path).rglob("*"))) > 0:
        print(f"  ✓ {label}: present")
    else:
        errors.append(f"{label} missing")

for tool in ["clang", "checkpolicy"]:
    p = subprocess.run(["which", tool], capture_output=True, text=True)
    print(f"  ✓ {tool}") if p.stdout.strip() else errors.append(f"{tool} not found")

if errors:
    print(f"\n❌ PREFLIGHT FAILED:")
    for e in errors: print(f"  • {e}")
    raise RuntimeError("Fix errors above")
else:
    print(f"\n✓ Preflight passed")

## 11. Apply scoring & output fixes
> Three bugs patched automatically:
> 1. `lstrip("**/")` → `removeprefix("**/")` — glob bug, HAL scores = 0.000
> 2. `_clean_output()` on backend sub-modules — markdown fences → SyntaxError
> 3. `output_root` on architect agent — HAL files in wrong directory

In [ ]:
# Fix 1: lstrip → removeprefix
for label, fpath in [("C3", f"{REPO_DIR}/multi_main_rag_dspy.py"),
                     ("C4", f"{REPO_DIR}/multi_main_c4_feedback.py")]:
    if not Path(fpath).exists(): continue
    src = Path(fpath).read_text()
    if 'pattern.lstrip("**/")' in src:
        Path(fpath).write_text(src.replace('pattern.lstrip("**/")', 'pattern.removeprefix("**/")'))
        print(f"  ✓ {label}: lstrip → removeprefix")
    else:
        print(f"  ✓ {label}: already patched")

# Fix 2: _clean_output() on backend sub-modules
be_path = f"{REPO_DIR}/agents/rag_dspy_backend_agent.py"
if Path(be_path).exists():
    be = Path(be_path).read_text()
    patched = False
    for old, new, tag in [
        ('model_content = getattr(result, "models_code", "") or ""\n                self._write',
         'model_content = getattr(result, "models_code", "") or ""\n                model_content = self._clean_output(model_content)\n                self._write', "model"),
        ('sim_content = getattr(result, "simulator_code", "") or ""\n                self._write',
         'sim_content = getattr(result, "simulator_code", "") or ""\n                sim_content = self._clean_output(sim_content)\n                self._write', "simulator"),
    ]:
        if old in be and f"_clean_output({tag}_content)" not in be:
            be = be.replace(old, new); patched = True
    if patched:
        Path(be_path).write_text(be); print("  ✓ Backend: _clean_output() added")
    else:
        print("  ✓ Backend: already patched")

# Fix 3: output_root on architect agent
for label, fpath in [("C3", f"{REPO_DIR}/multi_main_rag_dspy.py"),
                     ("C4", f"{REPO_DIR}/multi_main_c4_feedback.py")]:
    if not Path(fpath).exists(): continue
    src = Path(fpath).read_text()
    old = 'agent = RAGDSPyArchitectAgent(**AGENT_CFG)'
    new = 'agent = RAGDSPyArchitectAgent(**AGENT_CFG, output_root=str(OUTPUT_DIR))'
    if old in src:
        Path(fpath).write_text(src.replace(old, new))
        print(f"  ✓ {label}: output_root added")
    else:
        print(f"  ✓ {label}: architect already patched")

print("\n✓ All fixes applied")

## 12. Run C3 — RAG + DSPy optimised prompts
> RAG retrieval + DSPy-optimised prompts (~20 min).

In [ ]:
!rm -rf {REPO_DIR}/output_rag_dspy
start_ollama()
!python apply_chroma_fix.py
!python multi_main_rag_dspy.py

## 13. Run C4 — Feedback loop
> Generate → validate → refine, up to 3 retries per agent (~20 min).

In [ ]:
!rm -rf {REPO_DIR}/output_c4_feedback
start_ollama()
!python multi_main_c4_feedback.py

## 14. Reporting & analysis

In [ ]:
!python diagnose_outputs.py
!python rescore_all_conditions.py
!python compare_matched.py
!python analyze_final.py

In [ ]:
print("=" * 80)
print("MATCHED ANALYSIS RESULTS")
print("=" * 80)
!cat experiments/results/matched_analysis.md

## 15. Export & save to Drive

In [ ]:
from google.colab import files

artifacts = {
    "output_c1":     f"{REPO_DIR}/output",
    "output_c2":     f"{REPO_DIR}/output_adaptive",
    "output_c3":     f"{REPO_DIR}/output_rag_dspy",
    "output_c4":     f"{REPO_DIR}/output_c4_feedback",
    "dspy_programs": f"{REPO_DIR}/dspy_opt/saved",
}

for name, src_dir in artifacts.items():
    if Path(src_dir).exists():
        shutil.make_archive(f"/content/{name}", "zip", src_dir)
        shutil.copy(f"/content/{name}.zip", f"{DRIVE_SRC}/{name}.zip")
        print(f"  ✓ {name}.zip ({len(list(Path(src_dir).rglob('*')))} files)")
    else:
        print(f"  ⚠ {name}: not found")

shutil.copytree(f"{REPO_DIR}/experiments/results", f"{DRIVE_SRC}/experiments_results", dirs_exist_ok=True)
print("\n✓ All results saved to Drive")

for name in artifacts:
    zip_path = f"/content/{name}.zip"
    if Path(zip_path).exists():
        files.download(zip_path)

## 16. Utilities
> Run manually as needed.

In [ ]:
# Restart Ollama
# start_ollama()

In [ ]:
# Clean up
# shutil.rmtree(REPO_DIR, ignore_errors=True); print("✓ Repo removed")

In [ ]:
# Free disk (~300 MB)
# shutil.rmtree(AOSP_SRC_DIR, ignore_errors=True); print("✓ AOSP source removed")